In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

2026-06-25 22:56:43.799127: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv("../datasets/Churn_Modelling.csv")

In [3]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.1 MB


In [5]:
df.Geography.value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

## Creating a copy of the dataset to work with

In [6]:
df1 = df.copy()

## Deleting columns with too much unique entries

In [7]:
df1 = df1.drop(["RowNumber", "CustomerId", "Surname"], axis="columns")

In [8]:
df1.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## Converting Categorical columns to Numerical 

### Gender

In [9]:
df1['Gender'] = df1['Gender'].replace({"Female": 0, "Male": 1}).astype('int64')

In [10]:
df1.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


### Geography
Here I will be using one hot encoding

In [11]:
df1 = pd.get_dummies(df1, columns=['Geography'], dtype=int)

In [12]:
df1.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


## Identifying columns that needs scaling

In [13]:
cols_to_scale = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()


In [14]:
df1.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


### Train-Test_Split and scaling the columns


In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X = df1.drop("Exited", axis=1)
y = df1.Exited

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20) #included random state for reproducibility

In [18]:
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [19]:
X_train.shape

(8000, 12)

In [20]:
X_test.shape

(2000, 12)

## Defining a Neural network function

In [21]:
from sklearn.metrics import classification_report

In [22]:
def ANN(X_train, X_test, y_train, y_test, loss, epochs):
    tf.random.set_seed(42)
    model = keras.Sequential([
        keras.layers.Dense(30, input_dim = 12, activation = 'relu'),
        keras.layers.Dense(20, activation = 'relu'),
        keras.layers.Dense(10, activation = 'relu'),
        keras.layers.Dense(1, activation = 'sigmoid'),
    ])

    model.compile(optimizer='adam', loss=loss, metrics = ['accuracy'])
    model.fit(X_train, y_train, epochs= epochs)
    
    print("Accucacy", model.evaluate(X_test, y_test))
    y_preds = model.predict(X_test, verbose=0)
    y_preds = np.round(y_preds)

    print("Classification report \n", classification_report(y_test, y_preds))
    return y_preds

In [23]:
base_model = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy', 100)

/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7905 - loss: 0.4931
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7929 - loss: 0.4619
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7933 - loss: 0.4459
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8145 - loss: 0.4241
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8304 - loss: 0.3957
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8453 - loss: 0.3717
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8497 - loss: 0.3611
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8506 - loss: 0.3565
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8510 - loss: 0.3538
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8516 - loss: 0.3516
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8511 - loss: 0.3502
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━

## Creating a copy of the dataframe to work with

In [24]:
df2 = df1.copy()

In [25]:
df2.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1,0,0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0,0,1
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1,0,0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1,0,0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0,0,1


In [26]:
df2.Exited.value_counts()

Exited
0    7963
1    2037
Name: count, dtype: int64

### Seperating the values into classes based on count

In [27]:
minority_class = df2[df2.Exited == 1]
majority_class = df2[df2.Exited == 0]

In [28]:
minority_class.shape, majority_class.shape

((2037, 13), (7963, 13))

### Getting the number of values in each class

In [29]:
majority_count, minority_count = df2['Exited'].value_counts()

In [30]:
minority_count, majority_count

(2037, 7963)

## Handling imbalanced data by under-sampling the Majority class

In [31]:
undersampled_class = majority_class.sample(minority_count)
undersampled_df = pd.concat([undersampled_class, minority_class], axis=0)

In [32]:
undersampled_df.Exited.value_counts()

Exited
0    2037
1    2037
Name: count, dtype: int64

### Train test split and scaling on the undersampled dataframe

In [33]:
X_under = undersampled_df.drop('Exited', axis=1)
y_under = undersampled_df.Exited

In [34]:
X_train_under, X_test_under, y_train_under, y_test_under = train_test_split(X_under, y_under, test_size=0.2, random_state=20, stratify=y_under)

In [35]:
X_train_under[cols_to_scale] = scaler.fit_transform(X_train_under[cols_to_scale])
X_test_under[cols_to_scale] = scaler.transform(X_test_under[cols_to_scale])

In [36]:
undersampled_model = ANN(X_train_under, X_test_under, y_train_under, y_test_under, 'binary_crossentropy', 100)

/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5879 - loss: 0.6771
Epoch 2/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6456 - loss: 0.6364
Epoch 3/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6723 - loss: 0.6107
Epoch 4/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6855 - loss: 0.5896
Epoch 5/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6993 - loss: 0.5770
Epoch 6/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7100 - loss: 0.5664
Epoch 7/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7192 - loss: 0.5547
Epoch 8/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7312 - loss: 0.5407
Epoch 9/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7423 - loss: 0.5260
Epoch 10/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7499 - loss: 0.5121
Epoch 11/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7579 - loss: 0.5012
Epoch 12/100
102/102 ━━━━━━━━━━━━━━━━━━━━

### Observation from the under-sampled model
1. The Precision, Recall and F1 score for class 0 reduced significantly compared to the scores for the base model 
2. The Precision, Recall and F1 score for class 1 improved significantly compared to the scores for the base model and then the support is evenly distributed. However the Accuracy score dropped

## Handling imbalanced data by over-sampling the Minority class

### Resampling the dataset

In [37]:
oversampled_class = minority_class.sample(majority_count, replace=True)
oversampled_df = pd.concat([oversampled_class, majority_class], axis=0)

In [38]:
oversampled_df.Exited.value_counts()

Exited
1    7963
0    7963
Name: count, dtype: int64

### Train test split with the resampled dataframe

In [39]:
X_over = oversampled_df.drop('Exited', axis=1)
y_over = oversampled_df.Exited

In [40]:
X_train_over, X_test_over, y_train_over, y_test_over = train_test_split(X_over, y_over, test_size=0.2, random_state=20, stratify=y_over)

In [41]:
X_train_over[cols_to_scale] = scaler.fit_transform(X_train_over[cols_to_scale])
X_test_over[cols_to_scale] = scaler.transform(X_test_over[cols_to_scale])

### Developing a model with the resampled model

In [42]:
oversampled_model = ANN(X_train_over, X_test_over, y_train_over, y_test_over, 'binary_crossentropy', 100)

/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6581 - loss: 0.6182
Epoch 2/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7159 - loss: 0.5562
Epoch 3/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7423 - loss: 0.5187
Epoch 4/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7601 - loss: 0.4930
Epoch 5/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7684 - loss: 0.4789
Epoch 6/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7696 - loss: 0.4712
Epoch 7/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7726 - loss: 0.4662
Epoch 8/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7742 - loss: 0.4626
Epoch 9/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7762 - loss: 0.4596
Epoch 10/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7784 - loss: 0.4568
Epoch 11/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7797 - loss: 0.4542
Epoch 12/100
399/399 ━━━━━━━━━━━━━━━━━━━━

### Observation from the oversampled model
1. The Precision, Recall and F1 score for both class 0 and 1 improved compared to the under_sampled model
2. The model's accuracy is better than the under-sampled model but is less than the base model's accuracy
3. Oversampling handled the imbalance better than the under-sampled method

## Handling imbalanced data by SMOTE

In [43]:
from imblearn.over_sampling import SMOTE

In [44]:
smote = SMOTE(sampling_strategy='minority', random_state=42)

In [45]:
X = df2.drop('Exited', axis=1)
y = df2.Exited

### Train test split on the resampled dataframe and scaling the resampled data

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20, stratify=y)

In [47]:
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [48]:
X_train, y_train = smote.fit_resample(X_train,y_train)

In [49]:
y_train.value_counts()

Exited
1    6370
0    6370
Name: count, dtype: int64

In [50]:
sm_model = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy',100)

Epoch 1/100


/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6613 - loss: 0.6225
Epoch 2/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7149 - loss: 0.5616
Epoch 3/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7266 - loss: 0.5385
Epoch 4/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7410 - loss: 0.5172
Epoch 5/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7619 - loss: 0.4926
Epoch 6/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7745 - loss: 0.4733
Epoch 7/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7812 - loss: 0.4613
Epoch 8/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7852 - loss: 0.4533
Epoch 9/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7885 - loss: 0.4478
Epoch 10/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7886 - loss: 0.4439
Epoch 11/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7899 - loss: 0.4411
Epoch 12/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

### Observation from the SMOTE model
1. Unlike the other methods, my pipeline was quite different for this. I had to split the dataframe, scale and the resample just X_train and y_train.
2. The model performed quite well when predicting the "0" class but didn't do so well with the "1" class but then it performed better than the base model

## Handling Imbalanced data by Ensemble method

In [51]:
X = df2.drop('Exited', axis=1)
y = df2.Exited

In [52]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=20, stratify=y)

In [53]:
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

In [54]:
df3 = X_train.copy()
df3['Exited'] = y_train

In [55]:
df3_values0 = df3[df3.Exited == 0]
df3_values1 = df3[df3.Exited == 1]

In [56]:
df3_values0.shape, df3_values1.shape

((6370, 13), (1630, 13))

### Creating a batch function so I can have equal distribution in each class

In [57]:
def get_batch(df_majority, df_minority, start, end):
    df_ensemble = pd.concat([df_majority[start:end], df_minority])

    X_train = df_ensemble.drop('Exited', axis=1)
    y_train = df_ensemble.Exited

    return X_train, y_train
    

### Batch 1

In [58]:
X_train, y_train = get_batch(df3_values0, df3_values1, 0, 1630)

In [59]:
y_train.value_counts()

Exited
0    1630
1    1630
Name: count, dtype: int64

In [60]:
ensemble_model1 = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy',100)

/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6009 - loss: 0.6694
Epoch 2/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6531 - loss: 0.6331
Epoch 3/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6773 - loss: 0.6056
Epoch 4/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7006 - loss: 0.5802
Epoch 5/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7178 - loss: 0.5616
Epoch 6/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7301 - loss: 0.5487
Epoch 7/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7390 - loss: 0.5369
Epoch 8/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7479 - loss: 0.5251
Epoch 9/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7574 - loss: 0.5148
Epoch 10/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7644 - loss: 0.5055
Epoch 11/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7675 - loss: 0.4975
Epoch 12/100
102/102 ━━━━━━━━━━━━━━━━━━━━

### Batch 2

In [61]:
X_train, y_train = get_batch(df3_values0, df3_values1, 1630, 3620)

In [62]:
ensemble_model2 = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy', 100)

Epoch 1/100


/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


114/114 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.5434 - loss: 0.6852
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6320 - loss: 0.6486
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6735 - loss: 0.6119
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6953 - loss: 0.5883
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7086 - loss: 0.5710
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7218 - loss: 0.5575
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7293 - loss: 0.5457
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7362 - loss: 0.5345
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7417 - loss: 0.5239
Epoch 10/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7481 - loss: 0.5138
Epoch 11/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7577 - loss: 0.5046
Epoch 12/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step

### Batch 3

In [63]:
X_train, y_train = get_batch(df3_values0, df3_values1, 3620, 4890)

In [64]:
ensemble_model3 = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy',100)

Epoch 1/100


/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


91/91 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.5872 - loss: 0.6737
Epoch 2/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6676 - loss: 0.6323
Epoch 3/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6776 - loss: 0.6004
Epoch 4/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6872 - loss: 0.5866
Epoch 5/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7010 - loss: 0.5746
Epoch 6/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7141 - loss: 0.5614
Epoch 7/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7300 - loss: 0.5473
Epoch 8/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7376 - loss: 0.5321
Epoch 9/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7448 - loss: 0.5168
Epoch 10/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7510 - loss: 0.5022
Epoch 11/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7624 - loss: 0.4919
Epoch 12/100
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7655 - lo

### Batch 4

In [65]:
X_train, y_train = get_batch(df3_values0, df3_values1, 4890, 6370)

In [66]:
ensemble_model4 = ANN(X_train, X_test, y_train, y_test, 'binary_crossentropy', 100)

Epoch 1/100


/Users/mac/tensorflow_env/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.6293 - loss: 0.6577
Epoch 2/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6830 - loss: 0.6096
Epoch 3/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6942 - loss: 0.5904
Epoch 4/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7035 - loss: 0.5767
Epoch 5/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7141 - loss: 0.5646
Epoch 6/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7186 - loss: 0.5526
Epoch 7/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7273 - loss: 0.5404
Epoch 8/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7344 - loss: 0.5286
Epoch 9/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7402 - loss: 0.5181
Epoch 10/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7434 - loss: 0.5100
Epoch 11/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7492 - loss: 0.5035
Epoch 12/100
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7527 - lo

### Combining all the batches to get a final classification report

In [67]:
y_pred_final = ensemble_model1.copy()

for i in range(len(ensemble_model1)):
    n_ones = ensemble_model1[i] + ensemble_model2[i] + ensemble_model3[i] + ensemble_model4[i]
    if n_ones > 1:
        y_pred_final[i] = 1
    else:
        y_pred_final[i] = 0

In [68]:
from sklearn.metrics import classification_report

In [69]:
print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.94      0.71      0.81      1593
           1       0.42      0.81      0.55       407

    accuracy                           0.73      2000
   macro avg       0.68      0.76      0.68      2000
weighted avg       0.83      0.73      0.76      2000



### Observation from the ensembled model
1. The Recall and F1 score for class 1 is better than that of the base model even though the model didn't do that well predicting the "1" class.